# **LABORATORIO 7: REGRESIÓN LOGÍSTICA Y MÁQUINAS DE VECTOR DE SOPORTE**

## **PARTE A: Preparación de los datos**

### **Separación del Dataframe en variables numéricas y no numéricas**

In [1]:
# Importación de librerías
import pandas as pd # Trabajar con datos estructurados
import numpy as np
from IPython.display import display, HTML # Visualización de objetos
import matplotlib.pyplot as plt # Visualización de gráficos
import seaborn as sns # Visualización de gráficos
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
# Lectura de la data
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/breast-cancer-wisconsin.data"
data = pd.read_csv(url, header = None)
data.head()

,0,1,2,3,4,5,6,7,8,9,10
0,1000025,5,1,1,1,2,1,3,1,1,2
1,1002945,5,4,4,5,7,10,3,2,1,2
2,1015425,3,1,1,1,2,2,3,1,1,2
3,1016277,6,8,8,1,3,4,3,7,1,2
4,1017023,4,1,1,3,2,1,3,1,1,2


In [3]:
# Renombramiento de las columnas del dataframe
data.columns = ["id", "clump_thickness", "size_uniformity", "shape_uniformity", "marginal_adhesion", "epithelial_size",
               "bare_nucleoli", "bland_chromatin", "normal_nucleoli", "mitoses", "class"]
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 699 entries, 0 to 698
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id                 699 non-null    int64 
 1   clump_thickness    699 non-null    int64 
 2   size_uniformity    699 non-null    int64 
 3   shape_uniformity   699 non-null    int64 
 4   marginal_adhesion  699 non-null    int64 
 5   epithelial_size    699 non-null    int64 
 6   bare_nucleoli      699 non-null    object
 7   bland_chromatin    699 non-null    int64 
 8   normal_nucleoli    699 non-null    int64 
 9   mitoses            699 non-null    int64 
 10  class              699 non-null    int64 
dtypes: int64(10), object(1)
memory usage: 60.2+ KB


In [4]:
# Se transforman los datos para separar el dataframe en variables numéricas y no numéricas

# Variable 'bare_nucleoli':
# Se reemplazan los valores nulos (?) por la mediana
data['bare_nucleoli'] = data['bare_nucleoli'].replace("?",pd.NA)
mediana = data['bare_nucleoli'].median()
data['bare_nucleoli'] = data['bare_nucleoli'].fillna(mediana)

# Se convierte la variable a tipo entero
data['bare_nucleoli'] = data['bare_nucleoli'].astype(int)

# Variable 'class':
# Se modifican las categorías '2' y '4' por valores binarios: '0' corresponde a benigno y '1' a maligno
data['class'] = data['class'].apply(lambda x: 0 if x == 2 else 1)

# Se convierte la variable a tipo categórica
data['class'] = data['class'].astype('category')

# Variable 'id':
# Se convierte la variable a tipo cadena
data['id'] = data['id'].astype(str)

In [5]:
# Se separa el dataframe en variables numéricas y no numéricas
data_numeric = data.select_dtypes(include = 'number')
data_no_numeric = data.select_dtypes(exclude = 'number')

# Se muestra el dataframe de variables numéricas
display(HTML("<h4>DataFrame de variables numéricas</h4>"))
display(data_numeric.head())

# Se muestra el dataframe de variables no numéricas
display(HTML("<h4>DataFrame de variables no numéricas</h4>"))
display(data_no_numeric.head())

,clump_thickness,size_uniformity,shape_uniformity,marginal_adhesion,epithelial_size,bare_nucleoli,bland_chromatin,normal_nucleoli,mitoses
0,5,1,1,1,2,1,3,1,1
1,5,4,4,5,7,10,3,2,1
2,3,1,1,1,2,2,3,1,1
3,6,8,8,1,3,4,3,7,1
4,4,1,1,3,2,1,3,1,1


,id,class
0,1000025,0
1,1002945,0
2,1015425,0
3,1016277,0
4,1017023,0


### **Selección de variables predictoras según el Information Value (IV)**

In [6]:
# Se convierte la variable objetivo a un pd.Series

target = data_no_numeric['class'].squeeze()
type(target)

pandas.core.series.Series

In [7]:
def calc_iv(X, y, feature, bins=10):
    """
    Calcula el Information Value (IV) de una variable numérica respecto a un target binario (Series).

    Parámetros:
    -----------
    X : pd.DataFrame
        DataFrame que contiene las variables predictoras.
    
    y : pd.Series
        Serie binaria con los valores objetivo (0 = no evento, 1 = evento).
    
    feature : str
        Nombre de la columna de X sobre la cual se calculará el IV.
    
    bins : int, opcional (default = 10)
        Número de bins para discretizar la variable numérica usando cuantiles.

    Retorna:
    --------
    iv_value : float
        Valor del Information Value para la variable especificada.
    """
    # Crear DataFrame temporal con la variable y el target
    df_temp = pd.DataFrame({feature: X[feature], 'target': y})

    # Binning por cuantiles
    # Agrega una columna en el Dataframe temporal con el cuantil al que pertenece cada dato
    df_temp['bin'] = pd.qcut(df_temp[feature], q=bins, duplicates='drop') 

    # Agrupar por bin y calcular métricas
    # 'total' = cantidad total de eventos por bin, 'events' = cantidad de eventos a favor por bin, 'non_events' = cantidad de eventos en contra por bin
    # Se agrupan los datos por cuantiles, obteniendo por agregación conteo y suma de los eventos a favor (variable y: 'target')
    grouped = df_temp.groupby('bin')['target'].agg(
        total = 'count',
        events = lambda x: (x == 1).sum()
    )
    grouped['non_events'] = grouped['total'] - grouped['events'] # Se calcula la cantidad de eventos en contra por bin
    
    # Calcular tasas proporcionales a favor y en contra por bin
    grouped['event_rate'] = grouped['events'] / grouped['events'].sum() # Total de eventos a favor: grouped['events'].sum()
    grouped['non_event_rate'] = grouped['non_events'] / grouped['non_events'].sum() # Total de eventos en contra: grouped['non_events'].sum()

    # Calcular WoE de cada bin
    grouped['woe'] = np.log(grouped['event_rate'] / grouped['non_event_rate']).replace({np.inf: 0, -np.inf: 0})

    # Calcuar el IV por bin
    grouped['iv'] = (grouped['event_rate'] - grouped['non_event_rate']) * grouped['woe']

    return grouped['iv'].sum() # Devuelve el IV de la variable

In [8]:
# Se determina el IV de todas las variables
iv_dict = {}
for col in data_numeric.columns:
    try:
        iv = calc_iv(data_numeric, target, feature=col)
        iv_dict[col] = iv
    except Exception as e:
        print(f"Error con la columna {col}: {e}")

# Convertir a DataFrame para ordenarlo
iv_df = pd.DataFrame.from_dict(iv_dict, orient='index', columns=['IV'])
iv_df = iv_df.sort_values(by='IV', ascending=False)

# Se muestra el Dataframe con las variables y su Information Value (IV)
display(HTML("<h4>Information Value</h4>"))
display(iv_df.round(2))

# Filtrar las variables con IV >= 0.1 (predictivo débil o mayor)
selected_features = iv_df[iv_df['IV'] >= 0.1].index.tolist()
display(HTML("<h4>Las variables predictoras</h4>"))
display(selected_features)

# Filtrar el dataset
X_filtered = data_numeric[selected_features]

,IV
bare_nucleoli,4.60
size_uniformity,4.27
epithelial_size,3.83
bland_chromatin,3.32
shape_uniformity,3.21
marginal_adhesion,3.06
clump_thickness,2.15
normal_nucleoli,1.73
mitoses,0.72


['bare_nucleoli',
 'size_uniformity',
 'epithelial_size',
 'bland_chromatin',
 'shape_uniformity',
 'marginal_adhesion',
 'clump_thickness',
 'normal_nucleoli',
 'mitoses']

### **Separación en data de entrenamiento y prueba**

In [9]:
from sklearn.model_selection import train_test_split

# Se determina la variable objetivo y las variables predictoras
X = X_filtered # Variables numéricas predictoras
y = data_no_numeric['class'] # Variable objetivo categórica

# Se divide la data aleatoriamente en el 75% para el entrenamiento y 25% para la evaluación
# Se asigna un número semilla para la reproducibilidad (random_state)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

## **PARTE B: Modelado con Regresión Logística**

### **Modelo inicial de Regresión Logística**

In [10]:
import statsmodels.api as sm

# Agregar constante para el intercepto
X_train_sm = sm.add_constant(X_train)

# Ajustar el modelo
logit_model = sm.Logit(y_train.astype('int'), X_train_sm).fit()

# Mostrar resumen del modelo
print(logit_model.summary())

Optimization terminated successfully.
         Current function value: 0.082602
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                  class   No. Observations:                  524
Model:                          Logit   Df Residuals:                      514
Method:                           MLE   Df Model:                            9
Date:                Sun, 04 May 2025   Pseudo R-squ.:                  0.8726
Time:                        07:14:13   Log-Likelihood:                -43.283
converged:                       True   LL-Null:                       -339.63
Covariance Type:            nonrobust   LLR p-value:                7.732e-122
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -9.6225      1.206     -7.979      0.000     -11.986      -7.259
bare_nuc

### **Modelo con eliminación de variables no significativas**

In [11]:
# Filtrar variables con p-value < 0.05
significant_vars = logit_model.pvalues[logit_model.pvalues < 0.05].index.drop('const') # No se considera el intercepto

# Entrenar nuevo modelo con solo variables significativas
X_train_reduced = sm.add_constant(X_train[significant_vars])
logit_model_reduced = sm.Logit(y_train.astype('int'), X_train_reduced).fit()
print(logit_model_reduced.summary())

Optimization terminated successfully.
         Current function value: 0.106420
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:                  class   No. Observations:                  524
Model:                          Logit   Df Residuals:                      520
Method:                           MLE   Df Model:                            3
Date:                Sun, 04 May 2025   Pseudo R-squ.:                  0.8358
Time:                        07:14:13   Log-Likelihood:                -55.764
converged:                       True   LL-Null:                       -339.63
Covariance Type:            nonrobust   LLR p-value:                9.945e-123
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
const              -9.2081      0.992     -9.284      0.000     -11.152      -7.264
bare_nucleoli 

Al reducir de 9 a 3 variables, el modelo pierde algo de capacidad explicativa (pseudo R² baja de 0.8726 a 0.8358), pero simplifica el modelo sin perder significancia estadística (LLR p-value incluso mejor).

El aumento en el log-likelihood negativo (de -43.283 a -55.764) indica que el nuevo modelo explica un poco menos la variabilidad de los datos.

Esta es una buena señal de parsimonia: se logra un modelo más simple con pocas variables pero aún con un ajuste razonablemente bueno.

### **Cálculo de métricas de clasificación**

In [21]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Preparar los datos de prueba
X_test_reduced = sm.add_constant(X_test[significant_vars])

# Obtener probabilidades y predicciones
y_prob = logit_model_reduced.predict(X_test_reduced) # Probabilidad de pertenecer a la clase positiva
y_pred = (y_prob >= 0.5).astype(int) # Se genera una serie booleana de clases positivas (1) y negativas (0) con un umbral de 0.5, y se convierte a entero

# Calcular métricas
accuracy_logit = accuracy_score(y_test.astype(int), y_pred)
precision_logit = precision_score(y_test.astype(int), y_pred)
recall_logit = recall_score(y_test.astype(int), y_pred)
f1_logit = f1_score(y_test.astype(int), y_pred)

# Matriz de confusión para calcular especificidad
cm_logit = confusion_matrix(y_test.astype(int), y_pred)
tn, fp, fn, tp = cm.ravel()
specificity_logit = tn / (tn + fp)

# Mostrar resultados
print("Métricas del modelo Regresión Logística:")
print(f"Matriz de confusión:\n{cm_logit}")
print(f"Exactitud: {accuracy_logit:.2f}")
print(f"Precisión: {precision_logit:.2f}")
print(f"Sensibilidad (Recall): {recall_logit:.2f}")
print(f"Especificidad: {specificity_logit:.2f}")
print(f"F1 Score: {f1_logit:.2f}")


Métricas del modelo Regresión Logística:
Matriz de confusión:
[[116   2]
 [  5  52]]
Exactitud: 0.96
Precisión: 0.96
Sensibilidad (Recall): 0.91
Especificidad: 0.98
F1 Score: 0.94


- Alto rendimiento general: La exactitud del 96% indica que predice correctamente la gran mayoría de los casos.

- Baja tasa de falsos positivos: Con una precisión de 0.96 y una especificidad de 0.98, el modelo rara vez clasifica como positivo algo que no lo es.

- Buen recall (0.91): El modelo identifica correctamente la mayoría de los positivos reales, aunque pierde un pequeño porcentaje (9%).

- F1 Score alto (0.94): Muestra un excelente equilibrio entre detectar positivos y evitar errores en las predicciones positivas.

## **PARTE C: Modelo de SVM**

### **Entrenamiento del modelo SVM**

In [13]:
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

# Entrenar el modelo SVM con el conjunto de entrenamiento
svm_model = SVC(kernel='linear', random_state=42)  # Se usa el regresor lineal
svm_model.fit(X_train, y_train)

# Predecir sobre el conjunto de prueba
y_pred_svm = svm_model.predict(X_test)

### **Cálculo de métricas de clasificación**

In [19]:
# Calcular métricas
accuracy_svm = accuracy_score(y_test, y_pred_svm)
precision_svm = precision_score(y_test, y_pred_svm)
recall_svm = recall_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)
cm_svm = confusion_matrix(y_test, y_pred_svm)

print("Métricas del modelo SVM:")
print(f"Matriz de confusión:\n{cm_svm}")
print(f"Exactitud     : {accuracy_svm:.2f}")
print(f"Precisión     : {precision_svm:.2f}")
print(f"Sensibilidad  : {recall_svm:.2f}")
print(f"Puntuación F1 : {f1_svm:.2f}")

Métricas del modelo SVM:
Matriz de confusión:
[[116   2]
 [  4  53]]
Exactitud     : 0.97
Precisión     : 0.96
Sensibilidad  : 0.93
Puntuación F1 : 0.95


In [22]:
# Suponiendo que ya tienes estas métricas:
# accuracy_logit, precision_logit, recall_logit, f1_logit

print("\nComparación de modelos:")
print(f"{'Métrica':<15} {'Regresión Logística':<20} {'SVM':<10}")
print(f"{'Exactitud':<15} {accuracy_logit:<20.2f} {accuracy_svm:<10.2f}")
print(f"{'Precisión':<15} {precision_logit:<20.2f} {precision_svm:<10.2f}")
print(f"{'Sensibilidad':<15} {recall_logit:<20.2f} {recall_svm:<10.2f}")
print(f"{'Puntuación F1':<15} {f1_logit:<20.2f} {f1_svm:<10.2f}")


Comparación de modelos:
Métrica         Regresión Logística  SVM       
Exactitud       0.96                 0.97      
Precisión       0.96                 0.96      
Sensibilidad    0.91                 0.93      
Puntuación F1   0.94                 0.95      


Ambos modelos presentan muy buenos resultados en la clasificación de células cancerígenas y benignas; sin embago, el modelo de SVM presenta una ligera mejora en la métrica de sensibilidad, lo cual es importante en la detección de la clase positiva. Se recomendaría emplear el modelo de SVM considerando que en el contexto de detección de cancer, un falso negativo implicaría retrasos en tratamiento y consecuencias graves en la salud del paciente.